# 📘 Notebook 03: Indexes and Query Plans

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module A: Data at Rest · Notebook 3 of 4 in this module · (3 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a query on a large table can be slow, and what a **full table scan** is
- Describe an **index** and build a small one by hand
- Create indexes in SQLite and **measure** what they do to query time as a table grows
- Read a **query plan** to see whether the database is scanning or searching
- State what indexes **cost**, in space and in the speed of writes
- Recognise the queries that an index cannot help, and the case where a scan is the **better** choice

> **Prerequisites:** Notebooks 01 and 02. From *From Zero to Data Structures & Algorithms in Python*: Notebook 11 (Big O), Notebook 13 (binary search) and Notebook 18 (trees). This notebook does not re-teach them. It shows where a database uses them.
>
> 🔭 **Why this notebook matters:** this is the first time in the course that the data stops fitting, not in a table, but in **time**. A query that was instant on ten thousand rows takes a minute on a hundred million. Indexes are the reason databases can answer in milliseconds regardless, and they are the single most effective performance tool you will ever use. They are also widely misunderstood: people add them hopefully, without measuring, and are surprised when nothing improves or when writing becomes slow.

> 📦 **Libraries used in this notebook.**
>
> - **SQLite**, built into Python and introduced in Notebook 01.
> - **Matplotlib**, the standard plotting library, introduced in *From Python to Deep Learning & Fine-Tuning*, Notebook 07. We use it for one plot.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment.

In [ ]:
%pip install -q matplotlib

In [ ]:
import bisect
import random
import datetime
import sqlite3
import time

import matplotlib.pyplot as plt

The bookshop again, in a larger edition: 200,000 orders and about half a million order lines. Building it takes a few seconds.

In [ ]:
def make_shop(customers=1000, products=200, orders=10000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    customer_rows = [(i, f"Customer {i}", rng.choice(cities), 1950 + rng.randint(0, 55)) for i in range(1, customers + 1)]
    product_rows = [(i, f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(1, products + 1)]
    order_rows, item_rows = [], []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        order_rows.append((order_id, rng.randint(1, customers), day.isoformat()))
        for line in range(rng.randint(1, 4)):
            item_rows.append((len(item_rows) + 1, order_id, rng.randint(1, products), rng.randint(1, 3)))
    return customer_rows, product_rows, order_rows, item_rows

def build_database(customer_rows, product_rows, order_rows, item_rows):
    db = sqlite3.connect(":memory:")
    db.executescript("""
        CREATE TABLE customers   (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT, birth_year INTEGER);
        CREATE TABLE products    (product_id INTEGER PRIMARY KEY, title TEXT NOT NULL, category TEXT, price REAL);
        CREATE TABLE orders      (order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL, order_date TEXT);
        CREATE TABLE order_items (item_id INTEGER PRIMARY KEY, order_id INTEGER NOT NULL, product_id INTEGER NOT NULL, quantity INTEGER);
    """)
    db.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customer_rows)
    db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", product_rows)
    db.executemany("INSERT INTO orders VALUES (?, ?, ?)", order_rows)
    db.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", item_rows)
    db.commit()
    return db

customer_rows, product_rows, order_rows, item_rows = make_shop(customers=20000, orders=200000)
db = build_database(customer_rows, product_rows, order_rows, item_rows)

print(f"{len(customer_rows):,} customers, {len(order_rows):,} orders, {len(item_rows):,} order items")

## 1. Looking for a needle

### Intuition first
You want the orders of one particular customer. The `orders` table has 200,000 rows in no useful order. With nothing else to go on, there is only one way: read **every** row and check it. Like looking for a name in a telephone directory whose entries are in random order.

Reading a whole table to answer a query is called a **full table scan**. Its cost grows in step with the table: ten times the rows, ten times the wait.

Let us time it. A single query is too quick to measure reliably, so we run 200 of them, each for a different customer.

In [ ]:
random.seed(1)
wanted = [random.randint(1, len(customer_rows)) for i in range(200)]

def time_queries(db, sql, values):
    started = time.perf_counter()
    for value in values:
        db.execute(sql, (value,)).fetchall()
    return (time.perf_counter() - started) / len(values) * 1000        # milliseconds per query

orders_of_customer = "SELECT order_id, order_date FROM orders WHERE customer_id = ?"

scan_time = time_queries(db, orders_of_customer, wanted)
print(f"Without an index: {scan_time:.3f} milliseconds per query")

A few milliseconds does not sound like much. A busy website runs thousands of such queries every second, and this table is small by commercial standards.

Compare with a query that looks an order up by its **primary key**.

In [ ]:
by_key_time = time_queries(db, "SELECT customer_id, order_date FROM orders WHERE order_id = ?", wanted)
print(f"Lookup by primary key: {by_key_time:.4f} milliseconds per query")
print(f"That is about {scan_time / by_key_time:,.0f} times faster")

The same table, the same number of rows, and one lookup is thousands of times faster than the other. The database did not have to scan for the primary key. It keeps the table **organised** by that key, so it can go almost straight to the row. An index is the way to get the same benefit for any other column.

## 2. An index by hand

### Intuition first
A textbook has an index at the back: a list of terms **in alphabetical order**, each with the pages where it appears. You do not read the book to find *photosynthesis*. You look it up in the index, which is quick because the index is sorted, and then turn directly to the page.

A database index is the same thing: a separate, **sorted** copy of one column, in which each value is stored together with the location of its row.

Let us build one for `customer_id`.

In [ ]:
index = sorted((customer_id, position) for position, (order_id, customer_id, order_date) in enumerate(order_rows))

print("The first entries of the index (customer_id, position of the row):")
print(index[:8])

Because the index is sorted, all the entries of one customer sit next to each other, and we can find where they begin by **binary search**: look in the middle, discard half, repeat. Python's `bisect` module does it for us.

In [ ]:
def orders_by_scan(customer_id):
    return [row for row in order_rows if row[1] == customer_id]

def orders_by_index(customer_id):
    start = bisect.bisect_left(index, (customer_id, -1))         # binary search for the first entry
    result = []
    while start < len(index) and index[start][0] == customer_id:
        result.append(order_rows[index[start][1]])               # follow the pointer to the row
        start += 1
    return result

print(orders_by_index(4242)[:3])
print("Same rows either way:", sorted(orders_by_scan(4242)) == sorted(orders_by_index(4242)))

started = time.perf_counter()
for customer_id in wanted:
    orders_by_scan(customer_id)
hand_scan = (time.perf_counter() - started) / len(wanted) * 1000

started = time.perf_counter()
for customer_id in wanted:
    orders_by_index(customer_id)
hand_index = (time.perf_counter() - started) / len(wanted) * 1000

print(f"\nScan:  {hand_scan:.3f} ms per lookup")
print(f"Index: {hand_index:.4f} ms per lookup")

A scan of 200,000 rows examines 200,000 rows. A binary search of 200,000 sorted entries needs about 18 steps, because $2^{18}$ is roughly 260,000.

### Formal definition
An **index** on a column is an auxiliary data structure that maps each value of the column to the rows that contain it, organised so that a value can be found without examining them all. A scan costs $O(n)$. A lookup in a sorted index costs $O(\log n)$.

Real databases do not keep the index in one long sorted list, because inserting into the middle of a list is slow. They use a **B-tree**: a search tree, as in the companion course, in which each node holds hundreds of keys. The tree is therefore very shallow. Three or four levels are enough for hundreds of millions of rows, so any row is three or four steps away.

## 3. Indexes in the database

One statement creates an index. From then on the database keeps it up to date by itself and uses it whenever it helps. **The query does not change at all.**

In [ ]:
db.execute("CREATE INDEX orders_by_customer ON orders(customer_id)")

index_time = time_queries(db, orders_of_customer, wanted)
print(f"Without the index: {scan_time:.3f} ms per query")
print(f"With the index:    {index_time:.4f} ms per query")
print(f"About {scan_time / index_time:,.0f} times faster")

### Asking the database what it intends to do
How can we tell whether an index is being used, short of timing everything? We ask. Put `EXPLAIN QUERY PLAN` in front of any query, and the database describes its **plan** instead of running it.

In [ ]:
plan_number = 0

def plan(sql, db=db):
    global plan_number
    plan_number += 1
    # The numbered comment makes every request unique. Python remembers the answers to
    # statements it has seen before, and would otherwise show an out-of-date plan after we add or drop an index.
    for row in db.execute(f"EXPLAIN QUERY PLAN {sql} /* {plan_number} */"):
        print("  ", row[-1])

print("Orders of one customer:")
plan("SELECT order_id, order_date FROM orders WHERE customer_id = 4242")

print("\nOrders on one date:")
plan("SELECT order_id FROM orders WHERE order_date = '2024-03-15'")

Two words carry the message. **SEARCH** means the database goes directly to the rows it needs, here `USING INDEX orders_by_customer`. **SCAN** means it reads the whole table. There is no index on `order_date`, so the second query is a scan.

Reading the plan is the habit that separates people who tune databases from people who guess. It costs nothing, and it is exact.

### The effect of size
An index matters more as the table grows. Let us measure both kinds of lookup on tables of increasing size.

In [ ]:
sizes = [5000, 20000, 80000, 320000]
scan_times, search_times = [], []

for size in sizes:
    rng = random.Random(0)
    test_db = sqlite3.connect(":memory:")
    test_db.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT)")
    test_db.executemany("INSERT INTO orders VALUES (?, ?, '2024-01-01')",
                        ((i, rng.randint(1, size // 10)) for i in range(size)))
    values = [rng.randint(1, size // 10) for i in range(100)]

    scan_times.append(time_queries(test_db, "SELECT order_id FROM orders WHERE customer_id = ?", values))
    test_db.execute("CREATE INDEX by_customer ON orders(customer_id)")
    search_times.append(time_queries(test_db, "SELECT order_id FROM orders WHERE customer_id = ?", values))
    print(f"{size:>8,} rows   scan {scan_times[-1]:8.4f} ms   index {search_times[-1]:8.4f} ms")

plt.figure(figsize=(7, 4))
plt.loglog(sizes, scan_times, marker="o", label="Full scan: O(n)")
plt.loglog(sizes, search_times, marker="o", label="Index search: O(log n)")
plt.title("Time for one lookup as the table grows")
plt.xlabel("Rows in the table (log scale)")
plt.ylabel("Milliseconds per query (log scale)")
plt.legend()
plt.grid(True)
plt.show()

The scan line climbs in step with the size of the table. The index line is almost flat. On a table a thousand times larger still, the scan would take a thousand times longer and the indexed lookup a few steps more. This is the difference between $O(n)$ and $O(\log n)$, the one you plotted for linear and binary search in the companion course, now deciding whether a web page loads.

## 4. What an index costs

If indexes are so effective, why not put one on every column? Because an index is a second copy of data that must be **stored** and **kept up to date**.

Every time a row is inserted, each index on the table must be updated too. Let us measure the time to insert 200,000 rows into a table with no index, with one, and with three.

In [ ]:
def time_inserts(index_columns):
    test_db = sqlite3.connect(":memory:")
    test_db.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT)")
    for number, column in enumerate(index_columns):
        test_db.execute(f"CREATE INDEX index_{number} ON orders({column})")
    started = time.perf_counter()
    test_db.executemany("INSERT INTO orders VALUES (?, ?, ?)", order_rows)
    test_db.commit()
    elapsed = time.perf_counter() - started
    pages = test_db.execute("PRAGMA page_count").fetchone()[0]
    page_size = test_db.execute("PRAGMA page_size").fetchone()[0]
    return elapsed, pages * page_size / 1_000_000

print(f"{'indexes':<34} {'insert time':>12} {'size':>9}")
for columns in [[], ["customer_id"], ["customer_id", "order_date", "customer_id, order_date"]]:
    elapsed, megabytes = time_inserts(columns)
    print(f"{str(len(columns)) + ' ' + str(columns):<34} {elapsed:>10.2f} s {megabytes:>7.1f} MB")

Each index makes writing slower and the database larger. With three indexes, the table takes several times as long to fill and occupies several times the space.

So indexing is a **trade**: faster reads in exchange for slower writes and more storage. A table that is read constantly and changed rarely, such as a product catalogue, deserves several indexes. A table that receives thousands of new rows a second and is seldom queried, such as a log, deserves few.

## 5. When an index does not help

An index is a sorted list of the **values of a column**. It helps only when the query can be answered by looking a value up in that sorted list. Three common ways to prevent that follow.

### Wrapping the column in a function
We create an index on the date and ask for the orders of 2024 in two ways that mean the same thing.

In [ ]:
db.execute("CREATE INDEX orders_by_date ON orders(order_date)")

print("Using a function on the column:")
plan("SELECT COUNT(*) FROM orders WHERE SUBSTR(order_date, 1, 4) = '2024'")

print("\nUsing a range on the column itself:")
plan("SELECT COUNT(*) FROM orders WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01'")

The first plan says SCAN. The index is sorted by the full date, and the database cannot know how the results of `SUBSTR` are ordered, so it cannot jump to the right place. It has to go through every entry. (It reads the entries from the index and not from the table only because the index is the smaller of the two. That is what `COVERING INDEX` means: everything the query needs is in the index.) The second asks for a **range** of the column itself, and a range of a sorted list is one contiguous stretch: find where it starts, and read on to where it ends.

### A pattern that does not start at the beginning
A telephone directory helps you find the names that **begin** with *Pap*. It is no help at all in finding the names that **end** in *opoulos*.

In [ ]:
db.execute("CREATE INDEX customers_by_name ON customers(name COLLATE NOCASE)")

print("Names that start with a prefix:")
plan("SELECT * FROM customers WHERE name LIKE 'Customer 12%'")

print("\nNames that contain something anywhere:")
plan("SELECT * FROM customers WHERE name LIKE '%12%'")

### The wrong column of a combined index
An index can cover several columns. It is then sorted by the first, and within equal values of the first by the second, like a directory sorted by surname and then by first name. It can find *a surname*, or *a surname and a first name*. It cannot find *a first name* alone.

In [ ]:
db.execute("DROP INDEX orders_by_customer")
db.execute("DROP INDEX orders_by_date")
db.execute("CREATE INDEX orders_by_customer_and_date ON orders(customer_id, order_date)")

for description, condition in [("customer only", "customer_id = 4242"),
                               ("customer and date", "customer_id = 4242 AND order_date >= '2024-06-01'"),
                               ("date only", "order_date = '2024-06-01'")]:
    print(description + ":")
    plan(f"SELECT order_id FROM orders WHERE {condition}")

The first two conditions search the index. The third cannot use it to search: the dates are sorted only **within** each customer. The order of the columns in a combined index matters, and the rule is that the index serves queries that fix its **leading** columns.

> ⚠️ **Common pitfalls**
>
> - Applying a function or a calculation to an indexed column in `WHERE`. Rewrite the condition so that the bare column is compared with a value or a range.
> - Searching with `LIKE '%text%'`. No ordinary index can help. Searching inside text needs a different kind of index altogether.
> - Adding indexes without looking at the plan, and without measuring. An index that no query uses costs space and write time and gives nothing back.
> - Forgetting that the columns you **join** on need indexes too. A join looks a key up for every row, and without an index each lookup is a scan.

## 6. When a scan is the right answer

It is tempting to conclude that an index search always beats a scan. It does not, and the reason is worth understanding.

Following an index means: find the entry, then **jump** to the row it points to, for every matching row. That is quick when a query wants twenty rows out of a million. But suppose the query wants **most** of the table. Then the index sends the database jumping all over the table, one row at a time, to end up reading nearly all of it anyway. Reading the table straight through, in order, is faster.

Let us measure. The column `quantity` in `order_items` has only three different values, so `quantity = 1` matches about a third of the table. SQLite lets us force its hand either way: `INDEXED BY` insists on an index, and `NOT INDEXED` forbids one.

In [ ]:
db.execute("CREATE INDEX items_by_quantity ON order_items(quantity)")
db.execute("CREATE INDEX items_by_order ON order_items(order_id)")

def time_once(sql, repeats=5):
    started = time.perf_counter()
    for i in range(repeats):
        db.execute(sql).fetchall()
    return (time.perf_counter() - started) / repeats * 1000

cases = [
    ("few rows:   the lines of one order",  "order_id = 12345",  "items_by_order"),
    ("many rows:  every line with quantity 1", "quantity = 1",   "items_by_quantity"),
]
print(f"{'query':<40} {'rows':>8} {'scan':>10} {'index':>10}")
for description, condition, index_name in cases:
    rows = db.execute(f"SELECT COUNT(*) FROM order_items WHERE {condition}").fetchone()[0]
    scan = time_once(f"SELECT SUM(product_id) FROM order_items NOT INDEXED WHERE {condition}")
    search = time_once(f"SELECT SUM(product_id) FROM order_items INDEXED BY {index_name} WHERE {condition}")
    print(f"{description:<40} {rows:>8,} {scan:>8.2f} ms {search:>8.2f} ms")

Read the two rows. For the query that wants a handful of rows, the index wins by a very large factor. For the query that wants a third of the table, the **scan** is the faster one.

| | A few rows out of many | A large share of the table |
|---|---|---|
| What the index does | goes straight to them | jumps to a different place for every row |
| What the scan does | reads everything to find a few | reads everything, in order, once |
| Better choice | **index** | **scan** |

The property that decides is called **selectivity**: the share of the rows that a condition lets through. Indexes pay off for selective conditions. A real database estimates the selectivity of each condition from statistics it keeps about the data, and chooses between a scan and a search accordingly. That component is the **query optimiser**, and it is the reason you can write *what* you want and leave the *how* to the machine.

## 7. Where we stand

| The problem | The answer |
|---|---|
| Finding a few rows means reading all of them | an index: a sorted structure that leads straight to them |
| Not knowing why a query is slow | `EXPLAIN QUERY PLAN`: scan or search? |
| Indexes slow down writing and take space | index what is searched often, and no more |
| A query wants most of the table | a scan, which no index will beat |

So far every query has been a small, sharp question: the orders of one customer, the lines of one order. Systems built for such questions are tuned for exactly that. The next notebook is about the other kind of question: *what were the sales of each category in each month in each city?* Questions like that read most of the data, and they need a different design.

---
## ✏️ Exercises

### Exercise 1 (Scan or search?)
Without running them for time, use `plan` to find out whether each of these queries scans or searches. Then explain each answer in one sentence. (The indexes that exist at this point are on `orders(customer_id, order_date)`, `customers(name)`, `order_items(quantity)` and `order_items(order_id)`.)

In [ ]:
queries = [
    "SELECT * FROM orders WHERE order_id = 500",
    "SELECT * FROM orders WHERE customer_id = 77",
    "SELECT * FROM order_items WHERE product_id = 15",
    "SELECT * FROM customers WHERE city = 'Volos'",
]
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for query in queries:
    print(query)
    plan(query)
```

*The first searches by the primary key, which is always indexed. The second searches the combined index, because it fixes the index's leading column. The third and fourth scan: there is no index on `product_id` or on `city`.*
</details>

### Exercise 2 (Fix a slow query)
The query below finds the order lines of one product. Time it with `time_queries` for the 200 product ids in `product_ids`. Then create a suitable index, check the plan, and time it again.

In [ ]:
random.seed(2)
product_ids = [random.randint(1, 200) for i in range(200)]
lines_of_product = "SELECT item_id, quantity FROM order_items WHERE product_id = ?"
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
before = time_queries(db, lines_of_product, product_ids)
db.execute("CREATE INDEX items_by_product ON order_items(product_id)")
plan("SELECT item_id, quantity FROM order_items WHERE product_id = 15")
after = time_queries(db, lines_of_product, product_ids)

print(f"Before: {before:.3f} ms   After: {after:.3f} ms   ({before / after:.1f} times faster)")
```

*The improvement is real and much smaller than for the customer lookup in Section 3. Each product has about 2,500 order lines, so every query returns thousands of rows, each of which has to be fetched. The index removes the searching. It cannot remove the work of delivering a large result.*
</details>

### Exercise 3 (Rewrite the condition)
Each query below prevents the use of the index on `orders(customer_id, order_date)` or on `customers(name)`. Rewrite each so that the plan shows a SEARCH, without changing its meaning.

In [ ]:
slow_queries = [
    "SELECT order_id FROM orders WHERE customer_id + 0 = 4242",
    "SELECT order_id FROM orders WHERE customer_id = 4242 AND SUBSTR(order_date, 1, 7) = '2024-03'",
]
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for slow, fast in [
    (slow_queries[0], "SELECT order_id FROM orders WHERE customer_id = 4242"),
    (slow_queries[1], "SELECT order_id FROM orders WHERE customer_id = 4242 AND order_date >= '2024-03-01' AND order_date < '2024-04-01'"),
]:
    print("Before:"); plan(slow)
    print("After: "); plan(fast)
    same = sorted(db.execute(slow).fetchall()) == sorted(db.execute(fast).fetchall())
    print("Same result:", same, "\n")
```

*In the second query the original plan may already search on the customer, and then check the date of every one of that customer's orders. After the rewrite, the date range is part of the search itself. Always confirm that a rewritten query returns the same rows.*
</details>

### Exercise 4 (An index for a join)
Time the query below, which totals the quantity of books bought by each of 200 customers. It joins `orders` with `order_items`. Then drop the index `items_by_order`, time it again, and look at the plan in both cases. What was the index doing?

In [ ]:
quantity_of_customer = """
    SELECT SUM(i.quantity)
    FROM orders AS o
    JOIN order_items AS i ON i.order_id = o.order_id
    WHERE o.customer_id = ?
"""
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
with_index = time_queries(db, quantity_of_customer, wanted)
plan(quantity_of_customer.replace("?", "4242"))

db.execute("DROP INDEX items_by_order")
without_index = time_queries(db, quantity_of_customer, wanted[:20])
plan(quantity_of_customer.replace("?", "4242"))

print(f"\nWith the index on order_items(order_id): {with_index:.3f} ms")
print(f"Without it:                              {without_index:.3f} ms")
db.execute("CREATE INDEX items_by_order ON order_items(order_id)")
```

*For each of the customer's orders, the database must find that order's lines. With the index, each of those is a quick search. Without it, the database has to find another way, such as building a temporary index for the query or scanning half a million rows. Columns used to join tables are among the most valuable to index.*
</details>

### Exercise 5 (Where is the crossover? A little harder)
Section 6 showed an index losing to a scan when a third of the table matched. Find roughly where the two meet. Create a table of 300,000 rows with a column `bucket` holding random integers from 1 to 1,000, and an index on it. For conditions `bucket <= k` that select about 0.1%, 1%, 5%, 20% and 50% of the rows, time `SELECT SUM(payload) FROM t ... WHERE bucket <= k` with `INDEXED BY` and with `NOT INDEXED`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
rng = random.Random(0)
big = sqlite3.connect(":memory:")
big.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, bucket INTEGER, payload INTEGER)")
big.executemany("INSERT INTO t VALUES (?, ?, ?)", ((i, rng.randint(1, 1000), rng.randint(1, 100)) for i in range(300000)))
big.execute("CREATE INDEX t_by_bucket ON t(bucket)")

def timed(sql, repeats=3):
    started = time.perf_counter()
    for i in range(repeats):
        big.execute(sql).fetchall()
    return (time.perf_counter() - started) / repeats * 1000

print(f"{'share of rows':>13} {'scan':>10} {'index':>10}   faster")
for k in [1, 10, 50, 200, 500]:
    scan = timed(f"SELECT SUM(payload) FROM t NOT INDEXED WHERE bucket <= {k}")
    search = timed(f"SELECT SUM(payload) FROM t INDEXED BY t_by_bucket WHERE bucket <= {k}")
    print(f"{k / 10:>12.1f}% {scan:>8.2f} ms {search:>8.2f} ms   {'index' if search < scan else 'scan'}")
```

*The scan takes about the same time whatever the condition, since it always reads everything. The index time grows in step with the number of matching rows. Somewhere between a few percent and a few tens of percent the two lines cross. The exact point depends on the machine and the data, which is why databases estimate it and do not apply a fixed rule.*
</details>

## 🔑 Key takeaways

- Without an index, finding rows means a **full table scan**, whose cost grows with the table.
- An **index** is a sorted structure (in practice a B-tree) that maps the values of a column to their rows. A lookup costs $O(\log n)$.
- `CREATE INDEX` is all it takes. Queries do not change, and the database uses the index when it helps.
- `EXPLAIN QUERY PLAN` tells you whether a query **scans** or **searches**. Read it before and after every change.
- Indexes cost **storage** and slow down **writes**. Index what is searched often.
- An index cannot help when the column is wrapped in a function, when a pattern starts with a wildcard, or when a combined index's leading column is not fixed.
- For queries that return a large share of a table, a **scan is faster**. What decides is the **selectivity** of the condition.

---
**Next:** *Notebook 04: Data Warehouses*, where the questions change from "find this row" to "summarise everything", and the design of the tables changes with them.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*